# bronze_ingest

Ingesta de Posts y Users de StackOverflow (dataset de ClickHouse) a la capa Bronze.
Se guarda en Parquet con overwrite, en una carpeta por mes.

La fuente se lee con DuckDB. Probamos también con pyarrow, pero las filas de cada mes están repartidas por
todo el archivo y tocaba descargarlo casi completo: con Users fueron 5 minutos con DuckDB, y con pyarrow íbamos a un ritmo de unos 20.

In [0]:
%pip install duckdb==1.4.5


Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()


## Paso 1 – Librerías

In [0]:
from datetime import datetime

import duckdb
import pytz
from pyspark.sql.types import LongType, StringType, StructField, StructType, TimestampType

## Paso 2 – Parámetros

Se pueden cambiar desde el notebook o desde el Job. Los meses tienen que ser seguidos.

In [0]:
dbutils.widgets.text("anio", "2021")
dbutils.widgets.text("meses", "1,2")
dbutils.widgets.text("storage_account", "stsbmprod001")

anio = int(dbutils.widgets.get("anio"))
meses = [int(m) for m in dbutils.widgets.get("meses").split(",")]
storage_account = dbutils.widgets.get("storage_account")

if meses != list(range(meses[0], meses[-1] + 1)):
    raise ValueError(f"Los meses deben ser seguidos: {meses}")

bronze_base = f"abfss://bronze@{storage_account}.dfs.core.windows.net/stackoverflow"
url_fuente = "https://datasets-documentation.s3.eu-west-3.amazonaws.com/stackoverflow/parquet"
filas_por_lote = 50_000

spark.conf.set("spark.sql.session.timeZone", "UTC")
print(f"Año {anio}, meses {meses}")
print(bronze_base)

Año 2021, meses [1, 2]
abfss://bronze@stsbmprod001.dfs.core.windows.net/stackoverflow


## Paso 3 – Esquemas

In [0]:
esquema_posts = StructType([
    StructField("Id", LongType(), True),
    StructField("PostTypeId", LongType(), True),
    StructField("AcceptedAnswerId", LongType(), True),
    StructField("CreationDate", TimestampType(), True),
    StructField("Score", LongType(), True),
    StructField("ViewCount", LongType(), True),
    StructField("Body", StringType(), True),
    StructField("OwnerUserId", LongType(), True),
    StructField("OwnerDisplayName", StringType(), True),
    StructField("LastEditorUserId", LongType(), True),
    StructField("LastEditorDisplayName", StringType(), True),
    StructField("LastEditDate", TimestampType(), True),
    StructField("LastActivityDate", TimestampType(), True),
    StructField("Title", StringType(), True),
    StructField("Tags", StringType(), True),
    StructField("AnswerCount", LongType(), True),
    StructField("CommentCount", LongType(), True),
    StructField("FavoriteCount", LongType(), True),
    StructField("ContentLicense", StringType(), True),
    StructField("ParentId", StringType(), True),
    StructField("CommunityOwnedDate", TimestampType(), True),
    StructField("ClosedDate", TimestampType(), True),
])

esquema_users = StructType([
    StructField("Id", LongType(), True),
    StructField("Reputation", StringType(), True),
    StructField("CreationDate", TimestampType(), True),
    StructField("DisplayName", StringType(), True),
    StructField("LastAccessDate", TimestampType(), True),
    StructField("AboutMe", StringType(), True),
    StructField("Views", LongType(), True),
    StructField("UpVotes", LongType(), True),
    StructField("DownVotes", LongType(), True),
    StructField("WebsiteUrl", StringType(), True),
    StructField("Location", StringType(), True),
    StructField("AccountId", LongType(), True),
])

tablas = {
    "post": {"archivo": f"posts/{anio}.parquet", "esquema": esquema_posts},
    "users": {"archivo": "users.parquet", "esquema": esquema_users},
}

## Paso 4 – Leer la fuente con DuckDB

Posts viene en un archivo por año y Users en un solo archivo, por eso los dos se filtran por `CreationDate`.
Cada mes se escribe por lotes para no cargarlo completo en memoria: el primer lote sobrescribe la carpeta y los demás se agregan.

In [0]:
con = duckdb.connect()
con.execute("SET home_directory = '/tmp'")
con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")


def ruta_mes(tabla, mes):
    return f"{bronze_base}/{tabla}/{anio}/{mes:02d}"


def leer_mes(archivo, mes):
    inicio = f"{anio}-{mes:02d}-01"
    fin = f"{anio}-{mes + 1:02d}-01" if mes < 12 else f"{anio + 1}-01-01"
    consulta = f"""
        SELECT *
        FROM read_parquet('{url_fuente}/{archivo}', binary_as_string = true)
        WHERE CreationDate >= TIMESTAMPTZ '{inicio} 00:00:00+00'
          AND CreationDate < TIMESTAMPTZ '{fin} 00:00:00+00'
    """
    return con.execute(consulta).fetch_record_batch(filas_por_lote)


def ingestar_mes(tabla, archivo, esquema, mes):
    modo = "overwrite"
    total = 0
    for lote in leer_mes(archivo, mes):
        df = spark.createDataFrame(lote.to_pandas(), schema=esquema)
        df.coalesce(1).write.mode(modo).parquet(ruta_mes(tabla, mes))
        modo = "append"
        total += lote.num_rows
    return total

## Paso 5 – Guardar en Bronze

In [0]:
resultado = []

for tabla, config in tablas.items():
    for mes in meses:
        inicio = datetime.now()
        filas = ingestar_mes(tabla, config["archivo"], config["esquema"], mes)
        minutos = (datetime.now() - inicio).total_seconds() / 60
        print(f"{tabla} {anio}-{mes:02d}: {filas:,} filas en {minutos:.1f} min")
        if filas == 0:
            raise ValueError(f"No hay filas para {tabla} {anio}-{mes:02d}")
        resultado.append((tabla, mes, filas))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

post 2021-01: 325,977 filas en 3.0 min


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

post 2021-02: 304,043 filas en 0.4 min


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

users 2021-01: 194,555 filas en 1.4 min


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

users 2021-02: 180,574 filas en 0.4 min


## Paso 6 – Validar

Se vuelve a leer lo que quedó en Bronze y se compara con lo que se escribió. También se revisa que las fechas sean del mes.

In [0]:
validacion = []

for tabla, mes, filas in resultado:
    bronze = spark.read.parquet(ruta_mes(tabla, mes))
    leidas = bronze.count()
    primera, ultima = bronze.selectExpr("min(CreationDate)", "max(CreationDate)").first()
    validacion.append((tabla, mes, filas, leidas, str(primera), str(ultima)))

    if leidas != filas:
        raise ValueError(f"{tabla} {mes}: se escribieron {filas} filas y se leyeron {leidas}")
    if (primera.year, primera.month) != (anio, mes) or (ultima.year, ultima.month) != (anio, mes):
        raise ValueError(f"{tabla} {mes}: hay fechas fuera del mes ({primera} a {ultima})")

display(spark.createDataFrame(validacion, ["tabla", "mes", "escritas", "leidas", "primera_fecha", "ultima_fecha"]))

tabla,mes,escritas,leidas,primera_fecha,ultima_fecha
post,1,325977,325977,2021-01-01 00:00:01.337000,2021-01-31 23:59:57.690000
post,2,304043,304043,2021-02-01 00:00:21.210000,2021-02-28 23:59:45.160000
users,1,194555,194555,2021-01-01 00:00:11.187000,2021-01-31 23:58:46.487000
users,2,180574,180574,2021-02-01 00:00:09.357000,2021-02-28 23:58:17.387000


## Paso 7 – Ver los archivos en Bronze

`modificationTime` muestra cuándo se escribió cada archivo.

In [0]:
zona = pytz.timezone("America/Bogota")

for tabla, mes, _ in resultado:
    print(ruta_mes(tabla, mes))
    archivos = [
        (f.name, f.size, datetime.fromtimestamp(f.modificationTime / 1000, zona).strftime("%Y-%m-%d %H:%M:%S"))
        for f in dbutils.fs.ls(ruta_mes(tabla, mes))
    ]
    display(spark.createDataFrame(archivos, ["archivo", "tamano", "modificado"]))

print(f"Ingesta terminada: {datetime.now(zona).strftime('%Y-%m-%d %H:%M:%S')} (hora Colombia)")


abfss://bronze@stsbmprod001.dfs.core.windows.net/stackoverflow/post/2021/01


archivo,tamano,modificado
_SUCCESS,0,2026-11-04 20:54:47
_committed_1449011600913667188,124,2026-11-04 20:14:28
_committed_168959064422005217,123,2026-11-04 20:54:03
_committed_2490854498471160142,124,2026-11-04 20:53:16
_committed_2525686378634831068,124,2026-11-04 20:13:45
_committed_2899346356682305476,124,2026-11-04 20:15:45
_committed_3512050105625225781,124,2026-11-04 20:15:09
_committed_3697613090872915144,834,2026-11-04 20:52:36
_committed_4507183168786134285,124,2026-11-04 20:15:51
_committed_4961520032900315797,124,2026-11-04 20:54:46


abfss://bronze@stsbmprod001.dfs.core.windows.net/stackoverflow/post/2021/02


archivo,tamano,modificado
_SUCCESS,0,2026-11-04 20:55:09
_committed_1461662382477658490,124,2026-11-04 20:55:07
_committed_3118708928670798543,124,2026-11-04 20:16:03
_committed_4198446028852897495,124,2026-11-04 20:54:57
_committed_4385684070559700675,124,2026-11-04 20:15:59
_committed_4582152039298651656,834,2026-11-04 20:54:51
_committed_4979137379509615677,124,2026-11-04 20:16:17
_committed_5062324581628618358,124,2026-11-04 20:16:14
_committed_5902743171151121339,124,2026-11-04 20:54:54
_committed_6252373223474042584,124,2026-11-04 20:16:10


abfss://bronze@stsbmprod001.dfs.core.windows.net/stackoverflow/users/2021/01


archivo,tamano,modificado
_SUCCESS,0,2026-11-04 20:56:32
_committed_1911818389343039385,124,2026-11-04 20:56:31
_committed_223707555005717024,123,2026-11-04 20:56:22
_committed_4254536381091467807,124,2026-11-04 20:17:21
_committed_4449494764624767862,124,2026-11-04 20:17:23
_committed_7112212325277485231,124,2026-11-04 20:56:32
_committed_8316153260238541861,124,2026-11-04 20:16:52
_committed_8374244121628484888,124,2026-11-04 20:17:13
_committed_937925908227393702,533,2026-11-04 20:55:56
_committed_vacuum1500630205988292025,195,2026-11-04 20:55:57


abfss://bronze@stsbmprod001.dfs.core.windows.net/stackoverflow/users/2021/02


archivo,tamano,modificado
_SUCCESS,0,2026-11-04 20:56:55
_committed_249242674653745468,123,2026-11-04 20:17:45
_committed_361266192212622896,123,2026-11-04 20:17:34
_committed_5803448249221429385,124,2026-11-04 20:17:43
_committed_778834590447086561,123,2026-11-04 20:56:55
_committed_7935422869433014575,124,2026-11-04 20:17:41
_committed_8934565660542215141,124,2026-11-04 20:56:53
_committed_9178941700890172494,124,2026-11-04 20:56:51
_committed_941484525235879788,531,2026-11-04 20:56:43
_committed_vacuum6401825374398163007,193,2026-11-04 20:56:44


Ingesta terminada: 2026-11-04 20:57:06 (hora Colombia)
